In [0]:
# Configuration
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "Europe/Warsaw")
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("bronze_schema", "bronze", ["bronze", "gabrielajaniszews786_bronze"], "Bronze schema")
dbutils.widgets.combobox("silver_schema", "silver", ["silver", "gabrielajaniszews786_silver"], "Silver schema")
dbutils.widgets.dropdown("run_checks", "true", ["true", "false"], "Verification / experimentation")
dbutils.widgets.combobox("gold_schema", "gold", ["gold", "gabrielajaniszews786_gold"], "Gold schema")
RUN_CHECKS = dbutils.widgets.get("run_checks") == "true"
CATALOG       = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

BRONZE_SENSOR = f"{CATALOG}.{BRONZE_SCHEMA}.sensor_data"
VALID_SILVER_SENSOR = f"{CATALOG}.{SILVER_SCHEMA}.valid_sensor"
CHECKED_SILVER_SENSOR = f"{CATALOG}.{SILVER_SCHEMA}.checked_sensor"
QUARANTINE_SILVER_SENSOR = f"{CATALOG}.{SILVER_SCHEMA}.quarantine_sensor"
SENSOR_SILVER = f"{CATALOG}.{SILVER_SCHEMA}.sensor_silver"
BRONZE_PRICES = f"{CATALOG}.{SILVER_SCHEMA}.prices_bronze"
VALID_SILVER_PRICES = f"{CATALOG}.{SILVER_SCHEMA}.valid_prices"
QUARANTINE_SILVER_PRICES = f"{CATALOG}.{SILVER_SCHEMA}.quarantine_prices"





In [0]:
# Loading sensor data
bronze_df_sensor = spark.sql(f"SELECT * FROM {BRONZE_SENSOR}")
valid_df_sensor = spark.sql(f"SELECT * FROM {VALID_SILVER_SENSOR}")
quarantine_df_sensor = spark.sql(f"SELECT * FROM {QUARANTINE_SILVER_SENSOR}")
silver_sensor = spark.sql(f"SELECT * FROM {SENSOR_SILVER}")


In [0]:
# Loading prices
bronze_df_prices = spark.sql(f"SELECT * FROM {BRONZE_PRICES}")
valid_df_prices = spark.sql(f"SELECT * FROM {VALID_SILVER_PRICES}")
quarantine_df_prices = spark.sql(f"SELECT * FROM {QUARANTINE_SILVER_PRICES}")

In [0]:
# Function for reconciliation
def reconcile_counts(name, bronze_df, valid_df, quarantine_df, tolerance=0):
    bronze_n     = bronze_df.count()
    valid_n      = valid_df.count()
    quarantine_n = quarantine_df.count()
    gap = bronze_n - (valid_n + quarantine_n)
    if abs(gap) > tolerance:
        raise ValueError(
            f"[{name}] bronze {bronze_n:,} != valid {valid_n:,} + quarantine {quarantine_n:,} (gap {gap:,})")
    print(f"[{name}] OK: bronze {bronze_n:,} = valid {valid_n:,} + quarantine {quarantine_n:,}")
    return gap


In [0]:
# Function for deduplication
def dedup_reconciliation(valid_df, silver_df):
   distinct_valid = valid_df.select("event_id").distinct().count()
   silver_n = silver_df.count()
   if distinct_valid != silver_n:
       raise ValueError(f"[sensor dedup] distinct event_id {distinct_valid:,} != sensor_silver {silver_n:,}")
   return "deduplication valid"

In [0]:
# Loading gold tables for referential checks
consumption = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.consumption_hourly")
dim_datacenter = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_datacenter")
dim_date = spark.read.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_date")

In [0]:
# Function for silver > gold reconciliation
def reconcile_silver_gold(df_silver, df_gold):
    silver_groups = (df_silver
        .select("bidding_zone", "site_id",
                F.to_date("timestamp_utc").alias("date"),
                F.hour("timestamp_utc").alias("hour"))
        .distinct().count())
    fact_rows = df_gold.count()

    if silver_groups != fact_rows:
        raise ValueError(f"[silver->gold] distinct (zone,site,date,hour) in silver {silver_groups:,} "
                        f"!= fact rows {fact_rows:,}")
    print(f"[silver->gold] OK: {fact_rows:,} fact rows match distinct silver groups")

In [0]:
# Function for referential check
def referential_check(name, fact_df, fact_key, dim_df, dim_key):
    orphans = (fact_df.select(F.col(fact_key).alias("k")).distinct()
               .join(dim_df.select(F.col(dim_key).alias("k")).distinct(), "k", "left_anti")
               .count())
    if orphans > 0:
        raise ValueError(f"[{name}] {orphans:,} orphan keys in the fact are missing from the dimension")
    print(f"[{name}] OK: every {fact_key} exists in the dimension")
    return orphans

In [0]:
# Silver > gold reconciliation
reconcile_silver_gold(silver_sensor, consumption)

In [0]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Run all reconciliation checks and capture results
results = []

# 1. Sensor reconciliation
try:
    sensor_gap = reconcile_counts("sensor", bronze_df_sensor, valid_df_sensor, quarantine_df_sensor, tolerance=0)
    sensor_bronze_count = bronze_df_sensor.count()
    sensor_valid_count = valid_df_sensor.count()
    sensor_quarantine_count = quarantine_df_sensor.count()
    sensor_status = "PASSED"
    sensor_message = f"bronze {sensor_bronze_count:,} = valid {sensor_valid_count:,} + quarantine {sensor_quarantine_count:,}"
except Exception as e:
    sensor_status = "FAILED"
    sensor_message = str(e)
    sensor_bronze_count = bronze_df_sensor.count()
    sensor_valid_count = valid_df_sensor.count()
    sensor_quarantine_count = quarantine_df_sensor.count()

# 2. Prices reconciliation
try:
    bronze_df_prices = spark.sql(f"SELECT * FROM {BRONZE_PRICES}")
    valid_df_prices = spark.sql(f"SELECT * FROM {VALID_SILVER_PRICES}")
    quarantine_df_prices = spark.sql(f"SELECT * FROM {QUARANTINE_SILVER_PRICES}")
    prices_gap = reconcile_counts("prices", bronze_df_prices, valid_df_prices, quarantine_df_prices, tolerance=0)
    prices_bronze_count = bronze_df_prices.count()
    prices_valid_count = valid_df_prices.count()
    prices_quarantine_count = quarantine_df_prices.count()
    prices_status = "PASSED"
    prices_message = f"bronze {prices_bronze_count:,} = valid {prices_valid_count:,} + quarantine {prices_quarantine_count:,}"
except Exception as e:
    prices_status = "FAILED"
    prices_message = str(e)
    prices_bronze_count = None
    prices_valid_count = None
    prices_quarantine_count = None

# 3. Deduplication reconciliation
try:
    dedup_result = dedup_reconciliation(valid_df_sensor, silver_sensor)
    dedup_status = "PASSED"
    dedup_message = dedup_result
    distinct_valid_count = valid_df_sensor.select("event_id").distinct().count()
    silver_sensor_count = silver_sensor.count()
except Exception as e:
    dedup_status = "FAILED"
    dedup_message = str(e)
    distinct_valid_count = None
    silver_sensor_count = None

# 4. Silver to Gold reconciliation
try:
    reconcile_silver_gold(silver_sensor, consumption)
    silver_gold_status = "PASSED"
    silver_groups = (silver_sensor
        .select("bidding_zone", "site_id",
                F.to_date("timestamp_utc").alias("date"),
                F.hour("timestamp_utc").alias("hour"))
        .distinct().count())
    fact_rows = consumption.count()
    silver_gold_message = f"{fact_rows:,} fact rows match distinct silver groups"
except Exception as e:
    silver_gold_status = "FAILED"
    silver_gold_message = str(e)
    silver_groups = None
    fact_rows = None

# 5. Referential integrity check: site_id
try:
    referential_check("fact.site_id -> dim_datacenter", consumption, "site_id", dim_datacenter, "site_id")
    ref_site_status = "PASSED"
    ref_site_message = "every site_id exists in dim_datacenter"
except Exception as e:
    ref_site_status = "FAILED"
    ref_site_message = str(e)

# 6. Referential integrity check: date
try:
    referential_check("fact.date -> dim_date", consumption, "date", dim_date, "date")
    ref_date_status = "PASSED"
    ref_date_message = "every date exists in dim_date"
except Exception as e:
    ref_date_status = "FAILED"
    ref_date_message = str(e)

# Create reconciliation results DataFrame
reconciliation_results = spark.createDataFrame([
    ("sensor_reconciliation", sensor_status, sensor_message, sensor_bronze_count, sensor_valid_count, sensor_quarantine_count),
    ("prices_reconciliation", prices_status, prices_message, prices_bronze_count, prices_valid_count, prices_quarantine_count),
    ("dedup_reconciliation", dedup_status, dedup_message, distinct_valid_count, silver_sensor_count, None),
    ("silver_gold_reconciliation", silver_gold_status, silver_gold_message, silver_groups, fact_rows, None),
    ("referential_site_id", ref_site_status, ref_site_message, None, None, None),
    ("referential_date", ref_date_status, ref_date_message, None, None, None)
], ["check_name", "status", "message", "bronze_count", "valid_count", "quarantine_count"])

# Add timestamp column with local time and reorder columns
reconciliation_results = reconciliation_results.withColumn("reconciliation_date", F.current_timestamp()) \
    .select("reconciliation_date", "check_name", "status", "message", "bronze_count", "valid_count", "quarantine_count")

# Display results
print("\n" + "="*80)
print("RECONCILIATION RESULTS")
print("="*80)
display(reconciliation_results)

# Optionally save to a table for historical tracking
results_table = f"{CATALOG}.{SILVER_SCHEMA}.reconciliation_results"
reconciliation_results.write.mode("append").saveAsTable(results_table)
print(f"\nResults saved to {results_table}")

failed = reconciliation_results.filter("status = 'FAILED'").count()
if failed:
    raise Exception(f"{failed} reconciliation check(s) FAILED")